# Image-adaptive 3D LUT — FiveK reproduction

Settings: **GPU T4**, Internet **on**. Add Input → dataset `jesucristo/zeng-3lut-fivek-480p`.

To resume after a session ends: save this notebook's output as a dataset version, attach it as an input, and set `RESUME_FROM` below.

In [ ]:
REPO = "https://github.com/<your-username>/3d-lut.git"  # TODO: set after pushing
!rm -rf /kaggle/working/3d-lut && git clone -q $REPO /kaggle/working/3d-lut
%cd /kaggle/working/3d-lut

In [ ]:
import glob, os
hits = glob.glob('/kaggle/input/**/fiveK/test.txt', recursive=True)
assert hits, 'FiveK dataset not attached'
DATA = os.path.dirname(hits[0])
print(DATA)
for f in ('train_input.txt', 'train_label.txt', 'test.txt'):
    print(f, sum(1 for l in open(os.path.join(DATA, f)) if l.strip()))
print('inputs', len(os.listdir(f'{DATA}/input/JPG/480p')), 'targets', len(os.listdir(f'{DATA}/expertC/JPG/480p')))

In [ ]:
# sanity check: one input / expert C pair
from PIL import Image
import matplotlib.pyplot as plt
name = open(f'{DATA}/test.txt').readline().strip()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, sub in zip(ax, ('input', 'expertC')):
    a.imshow(Image.open(f'{DATA}/{sub}/JPG/480p/{name}.jpg')); a.set_title(f'{sub} {name}'); a.axis('off')

In [ ]:
!python -m pytest -q

In [ ]:
RUN = '/kaggle/working/runs/fivek'
RESUME_FROM = None  # e.g. '/kaggle/input/<your-output-dataset>/runs/fivek'
if RESUME_FROM:
    os.makedirs(RUN, exist_ok=True)
    !cp {RESUME_FROM}/*.pt {RESUME_FROM}/log.jsonl {RUN}/

In [ ]:
# paper settings: batch 1, lr 1e-4, 33^3 LUTs x3. Checkpoints every epoch; re-run to resume.
!python train.py --data {DATA} --out {RUN} --epochs 400 --batch-size 1 --workers 4 --eval-every 10

In [ ]:
!python eval.py --data {DATA} --ckpt {RUN}/best.pt --save-images